# Map Matching & Subgraph Extraction

1. **Map Matching**: Chuyển đổi dữ liệu Lịch sử từ `(stop_A, stop_B)` thành các cạnh `(u, v)` của đồ thị OSM. Xử lý xung đột khi nhiều tuyến xe buýt cùng chạy qua 1 con đường.
2. **Data-Driven Filter (Subgraph)**: Cắt tỉa đồ thị 561,000 edges xuống còn một 'Bộ xương sống' siêu nhẹ (Chỉ chứa đường lớn + hẻm có xe buýt chạy).

In [1]:
import pandas as pd
import pickle
import json
import networkx as nx
from collections import defaultdict
import os

## 1. Load Dữ liệu
Bao gồm: Đồ thị gốc, Từ điển Mapping (segment_lengths), và Lịch sử Baseline.

In [2]:
graph_file = r'../data/hcmc_routing_brain_v2.pkl'
segment_file = r'../data/segment_lengths_v2.json'
baseline_file = r'../data/historical_baseline.pkl'

with open(graph_file, 'rb') as f:
    G = pickle.load(f)
print(f"Đồ thị V2: {len(G.nodes):,} nodes, {len(G.edges):,} edges")

with open(segment_file, 'r', encoding='utf-8') as f:
    segment_lengths = json.load(f)
print(f"Segment Mapping: {len(segment_lengths):,} đoạn")

with open(baseline_file, 'rb') as f:
    baseline_dict = pickle.load(f)
print(f"Historical Baseline: {len(baseline_dict):,} bản ghi")

Đồ thị V2: 275,940 nodes, 561,808 edges
Segment Mapping: 4,267 đoạn
Historical Baseline: 728,720 bản ghi


## 2. Lọc Đồ thị con (Data-Driven Subgraph)
Sử dụng biến `is_bus_route` đã được nhúng sẵn trong đồ thị V2 để lọc nhanh chóng.

In [3]:
# Lọc đồ thị
major_highways = {'trunk', 'primary', 'secondary', 'tertiary', 'trunk_link', 'primary_link', 'secondary_link', 'tertiary_link'}
edges_to_keep = []

for u, v, k, data in G.edges(keys=True, data=True):
    hw = data.get('highway', 'unknown')
    if isinstance(hw, list):
        hw = hw[0]
    
    # Giữ lại nếu là đường lớn HOẶC là đường xe buýt (dựa vào cờ is_bus_route đã được bake)
    if hw in major_highways or data.get('is_bus_route', False):
        edges_to_keep.append((u, v, k))

subgraph = G.edge_subgraph(edges_to_keep).copy()
print(f"Đồ thị sau khi lọc: {len(subgraph.nodes):,} nodes, {len(subgraph.edges):,} edges")
print("Giảm được:", round((1 - len(subgraph.edges)/len(G.edges)) * 100, 2), "% khối lượng")

with open('../data/hcmc_routing_brain_v2_subgraph.pkl', 'wb') as f:
    pickle.dump(subgraph, f)

Đồ thị sau khi lọc: 68,805 nodes, 103,662 edges
Giảm được: 81.55 % khối lượng


## 3. Map Matching Baseline -> Đồ thị (Edge-based Baseline)
Chuyển từ `(stop_A, stop_B)` sang `(u, v)` để thuật toán A* dễ dàng tra cứu. 
Nếu nhiều segment chồng lên nhau trên 1 cạnh (u, v), ta lấy trung bình cộng tốc độ.

In [4]:
# Dictionary tạm thời lưu danh sách tốc độ cho mỗi (u, v, day_type, time_slot)
edge_speed_aggregator = defaultdict(list)

for (stop_u, stop_v, day_type, time_slot), speed in baseline_dict.items():
    seg_id = f"{stop_u}_{stop_v}"
    seg_data = segment_lengths.get(seg_id)
    if not seg_data:
        continue
        
    nodes = seg_data.get('osmnx_nodes', [])
    for i in range(len(nodes) - 1):
        u, v = nodes[i], nodes[i+1]
        # Gán tốc độ của segment cho cạnh (u, v)
        edge_speed_aggregator[(u, v, day_type, time_slot)].append(speed)

# Tính trung bình
edge_baseline_dict = {}
for key, speeds in edge_speed_aggregator.items():
    # key = (u, v, day_type, time_slot)
    edge_baseline_dict[key] = round(sum(speeds) / len(speeds), 2)

print(f"Hoàn tất Map Matching! Thu được {len(edge_baseline_dict):,} bản ghi Lịch sử cấp độ Edge (Cạnh).")

output_file = r'../data/edge_historical_baseline.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(edge_baseline_dict, f)
print(f"Dung lượng file Edge Baseline: {os.path.getsize(output_file) / 1024 / 1024:.2f} MB")

Hoàn tất Map Matching! Thu được 12,261,763 bản ghi Lịch sử cấp độ Edge (Cạnh).
Dung lượng file Edge Baseline: 343.34 MB
